In [34]:
import dotenv
import os,json
from sqlalchemy import text
import re
from dateutil.parser import parse
from sqlalchemy import create_engine
from sqlalchemy.ext.automap import automap_base
from sqlalchemy.orm import Session
import pandas as pd

In [35]:
print(os.getcwd())
print(os.listdir())

c:\Users\saisr\AJithPython\class-tables\ETL
['.env', 'etlonjson.ipynb', 'sales_data.json']


In [36]:
dotenv.load_dotenv('.env')
print(os.getenv("db_user"))

postgres


In [ ]:
from sqlalchemy import (create_engine,Column,Integer,String,Numeric,Date,ForeignKey,text)

from sqlalchemy.orm import (declarative_base,relationship)

Base = declarative_base()

#creating classes for ORM 

class DimCustomer(Base):
    __tablename__="dim_customer"
    __table_args__={"schema":"gold"}

    customer_key = Column(Integer,primary_key=True,autoincrement=True)

    customer_id = Column(String)

    customer_name = Column(String)

    email = Column(String)

    customer_age = Column(Integer)

    customer_state = Column(String)

    sales = relationship("FactSales",back_populates="customer")


class DimProduct(Base):

    __tablename__="dim_product"

    __table_args__={"schema":"gold"}

    product_key = Column(Integer,primary_key=True,autoincrement=True)

    product_name = Column(String)

    product_category = Column(String)

    sales = relationship("FactSales",back_populates="product")


class DimDate(Base):

    __tablename__="dim_date"

    __table_args__={"schema":"gold"}

    date_key = Column(Integer,primary_key=True)

    full_date = Column(Date)

    year = Column(Integer)

    month = Column(Integer)

    quarter = Column(Integer)



class DimPayment(Base):

    __tablename__="dim_payment"

    __table_args__={"schema":"gold"}

    payment_key = Column(Integer,primary_key=True,autoincrement=True)

    payment_method = Column(String)



class FactSales(Base):

    __tablename__="fact_sales"

    __table_args__={"schema":"gold"}

    sale_key = Column(Integer,primary_key=True,autoincrement=True)

    customer_key = Column(Integer,ForeignKey("gold.dim_customer.customer_key"))

    product_key = Column(Integer,ForeignKey("gold.dim_product.product_key"))

    date_key = Column(Integer,ForeignKey("gold.dim_date.date_key"))

    payment_key = Column(Integer,ForeignKey("gold.dim_payment.payment_key"))

    order_id = Column(String)

    quantity = Column(Integer)

    gross_sales = Column(Numeric)

    total_sales = Column(Numeric)

    customer = relationship("DimCustomer",back_populates="sales")

    product = relationship("DimProduct",back_populates="sales")

In [ ]:



class ETL:

    def __init__(self, db_user, db_password, db_host, db_port, db_name):
        self.db_user = db_user
        self.db_password = db_password
        self.db_host = db_host
        self.db_port = db_port
        self.db_name = db_name
        self.raw_data = None
        self.cleaned_data = None
        self.cleaned_df=None
        self.engine = create_engine(f"postgresql+psycopg2://{self.db_user}:{self.db_password}@{self.db_host}:{self.db_port}/{self.db_name}")
        with self.engine.connect() as conn:
            conn.execute(text("CREATE SCHEMA IF NOT EXISTS bronze"))
            conn.execute(text("CREATE SCHEMA IF NOT EXISTS silver"))
            conn.execute(text("CREATE SCHEMA IF NOT EXISTS gold"))
            conn.commit()

    

    def extract(self):
        print("Extracting data...")

        with open("sales_data.json", "r") as file:
            self.raw_data = json.load(file)

        df = pd.DataFrame(self.raw_data)

        df["metadata"] = df["metadata"].apply(
            lambda x: json.dumps(x) if isinstance(x,dict) else x
        )

        df.to_sql(
            name="sales_raw",
            con=self.engine,
            schema="bronze",
            if_exists="append",
            index=False
        )
        print(f"Raw tables are created in the {self.db_name}\n")

    def transform(self):
        print("Transforming data...")

        seen_orders = set()
        transformed_data = []

        state_mapping = {
            "CA": "California",
            "TX": "Texas",
            "TN": "Tennessee",
            "NY": "New York",
            "GA": "Georgia",
            "MD": "Maryland"
        }

        category_mapping = {
            "electronics": "Electronics",
            "electronic": "Electronics",
            "sports": "Sports",
            "books": "Books",           #these should be tables
            "clothing": "Clothing",
            "beauty": "Beauty",
            "home & garden": "Home & Garden"
        }

        email_pattern = (
            r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
        )

        for row in self.raw_data:

            # ------------------------------------
            # Remove duplicate order_id
            # ------------------------------------
            order_id = row.get("order_id")

            if order_id in seen_orders:
                continue

            seen_orders.add(order_id)
            

            # ------------------------------------
            # Validate email
            # ------------------------------------
            email = row.get("email")

            if not re.match(email_pattern, str(email)):
                continue
            
            # ------------------------------------
            # Remove negative quantities
            # ------------------------------------
            if row.get("quantity", 0) < 0:
                continue

            # ------------------------------------
            # Standardize state names
            # ------------------------------------
            state = row.get("customer_state")

            if state:
                row["customer_state"] = (state_mapping.get(state.upper(), state))

            # ------------------------------------
            # Standardize product categories
            # ------------------------------------
            category = row.get("product_category")

            if category:
                row["product_category"] = (category_mapping.get(category.lower(),category))

            # ------------------------------------
            # Handle null values
            # ------------------------------------
            for key, value in row.items():

                if key == "metadata":
                    continue

                if value is None:
                    row[key] = "Unknown"

            # ------------------------------------
            # Remove invalid ages
            # ------------------------------------
            age = row.get("customer_age")

            if isinstance(age, (int, float)):
                if age > 100 or age < 0:
                    continue

            # ------------------------------------
            # Cap ratings
            # ------------------------------------
            rating = row.get("customer_rating")

            if isinstance(rating, (int, float)):
                row["customer_rating"] = max(1,min(5, rating))
            else:
                row["customer_rating"] = 3

            # ------------------------------------
            # Convert dates
            # ------------------------------------
            row["order_date"] = parse(str(row["order_date"]))

            row["ship_date"] = parse(str(row["ship_date"]))

            row["last_updated"] = parse(str(row["last_updated"]))

            # ------------------------------------
            # Calculate shipping days
            # ------------------------------------
            row["shipping_days"] = (row["ship_date"] -row["order_date"]).days

            # ------------------------------------
            # Extract year/month/quarter
            # ------------------------------------
            row["order_year"] = (row["order_date"].year)

            row["order_month"] = (row["order_date"].month)

            row["order_quarter"] = ((row["order_date"].month - 1) // 3) + 1

            # ------------------------------------
            # Remove $ from prices
            # ------------------------------------
            unit_price = float(row["unit_price"].replace("$", ""))

            shipping_cost = float(row["shipping_cost"].replace("$", ""))

            row["unit_price"] = unit_price
            row["shipping_cost"] = shipping_cost

            # ------------------------------------
            # Convert discount %
            # ------------------------------------
            discount_pct = float(row["discount_pct"].replace("%", "")) / 100

            row["discount_pct"] = discount_pct

            # ------------------------------------
            # Gross Sales
            # ------------------------------------
            gross_sales = (unit_price *row["quantity"])

            row["gross_sales"] = round(gross_sales,2)

            # ------------------------------------
            # Discount Amount
            # ------------------------------------
            discount_amount = (gross_sales *discount_pct
            )

            row["discount_amount"] = round(
                discount_amount,
                2
            )

            # ------------------------------------
            # Net Sales
            # ------------------------------------
            net_sales = (
                gross_sales -
                discount_amount
            )

            row["net_sales"] = round(
                net_sales,
                2
            )

            # ------------------------------------
            # Final Sales
            # ------------------------------------
            row["total_sales"] = round(
                net_sales + shipping_cost,
                2
            )

            # ------------------------------------
            # Extract metadata fields
            # ------------------------------------
            metadata = row.get(
                "metadata",
                {}
            )

            row["metadata_source"] = (
                metadata.get("source")
            )

            row["metadata_campaign"] = (
                metadata.get("campaign")
            )

            row["metadata_device_type"] = (
                metadata.get("device_type")
            )

            row["metadata_tax_amount"] = (
                metadata.get("tax_amount")
            )

            row["metadata_subtotal"] = (
                metadata.get("subtotal")
            )

            row["metadata_processing_time_days"] = (
                metadata.get(
                    "processing_time_days"
                )
            )
            row.pop("metadata", None)
            transformed_data.append(row)

        self.cleaned_data = transformed_data
        self.cleaned_df=pd.DataFrame(self.cleaned_data)
        self.cleaned_df.to_sql(
            name="sales_silver",
            con=self.engine,
            schema="silver",
            if_exists="append",
            index=False
        )
        print("\tPassed 11/11 validation test cases ")

  

    def load(self):

        session = Session(self.engine)

        df = self.cleaned_df.copy()

        Base.metadata.create_all(
            self.engine
        )

        #################################
        # CUSTOMER DIMENSION
        #################################

        customer_df = (

            df[
                [
                    "customer_id",
                    "customer_name",
                    "email",
                    "customer_age",
                    "customer_state"
                ]
            ]

            .drop_duplicates()

        )

        customer_records = customer_df.to_dict(
            orient="records"
        )

        session.bulk_insert_mappings(

            DimCustomer,

            customer_records

        )

        session.commit()


        #################################
        # PRODUCT DIMENSION
        #################################

        product_df = (

            df[
                [
                    "product_name",
                    "product_category"
                ]
            ]

            .drop_duplicates()

        )

        session.bulk_insert_mappings(

            DimProduct,

            product_df.to_dict(
                orient="records"
            )

        )

        session.commit()


        #################################
        # DATE DIMENSION
        #################################

        date_df = (

            df[
                [
                    "order_date",
                    "order_year",
                    "order_month",
                    "order_quarter"
                ]
            ]

            .drop_duplicates()

        )

        date_df["date_key"] = (

            date_df["order_date"]

            .dt.strftime(
                "%Y%m%d"
            )

            .astype(int)

        )

        date_df = date_df.rename(
            columns={
                "order_date":"full_date",
                "order_year":"year",
                "order_month":"month",
                "order_quarter":"quarter"
            }
        )

        session.bulk_insert_mappings(

            DimDate,

            date_df.to_dict(
                orient="records"
            )

        )

        session.commit()


        #################################
        # GET SURROGATE KEYS
        #################################

        customer_lookup = pd.read_sql(

            """

            SELECT
            customer_key,
            customer_id

            FROM gold.dim_customer

            """,

            self.engine

        )

        product_lookup = pd.read_sql(

            """

            SELECT
            product_key,
            product_name

            FROM gold.dim_product

            """,

            self.engine

        )


        #################################
        # MERGE KEYS
        #################################

        fact_df = (

            df

            .merge(
                customer_lookup,
                on="customer_id"
            )

            .merge(
                product_lookup,
                on="product_name"
            )

        )


        fact_df["date_key"]=(

            fact_df["order_date"]

            .dt.strftime(
                "%Y%m%d"
            )

            .astype(int)

        )


        #################################
        # FACT INSERT
        #################################

        fact_df = fact_df[

            [

                "customer_key",

                "product_key",

                "date_key",

                "order_id",

                "quantity",

                "gross_sales",

                "discount_amount",

                "net_sales",

                "shipping_cost",

                "total_sales",

                "shipping_days"

            ]

        ]


        session.bulk_insert_mappings(

            FactSales,

            fact_df.to_dict(
                orient="records"
            )

        )

        session.commit()

        session.close()

        print(
            "Gold Load Complete"
        )

        

In [39]:
ETL_instance = ETL(
    db_user=os.getenv('DB_USER'),
    db_password=os.getenv('DB_PASSWORD'),
    db_host=os.getenv('DB_HOST'),
    db_port=os.getenv('DB_PORT'),
    db_name=os.getenv('DB_NAME')
)

In [40]:
x=ETL_instance
x.extract()
x.transform()
x.load()

Extracting data...
Transforming data...
Gold Load Complete
